# Tutorial 14: Hybrid Optimization Strategies

**Prerequisites:** Tutorial 12 — Differential Evolution, Tutorial 13 — Analytical Gradients  
**Up Next:** None (capstone)

---

## Concept

The tutorials so far have revealed a fundamental tension:

- **Global methods** (DE, PSO) explore the full design space and find the right *basin*, but converge slowly to the exact optimum.
- **Local methods** (BFGS, Newton) converge fast once nearby a minimum, but get trapped in local minima if started far away.

The practical solution used in most engineering optimization is a **hybrid strategy**: run a global method to identify a promising region, then switch to a local method to polish the result. This capstone tutorial implements and benchmarks that approach.

## Key Takeaway

> **In practice, hybrid global+local strategies are the standard approach for engineering optimization. Use DE to find the right basin, then BFGS with analytical gradients to converge to the precise optimum.**

## Math Core

There is no new math here — the power is in the *combination*. The hybrid approach uses:

1. **Differential Evolution** (Tutorial 12): population-based global search, $O(N_\text{pop} \times N_\text{gen})$ function evaluations
2. **L-BFGS-B with analytical gradients** (Tutorial 13): superlinear local convergence, exact gradients at $O(1)$ cost per evaluation

The hybrid cost is:

$$C_\text{hybrid} = C_\text{DE}(N_\text{pop}, G_\text{coarse}) + C_\text{BFGS}(k_\text{polish})$$

where $G_\text{coarse} \ll G_\text{full}$ (we stop DE early) and $k_\text{polish}$ is typically small because BFGS starts near the optimum.

## Code

### Setup: the four-bar problem (same as all tutorials)

In [ ]:
from dms.mechanisms.fourbar import FourBar
from dms.curves import CompareCurves
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution
import time
%matplotlib inline

# Fixed link lengths
L0, L1 = 1.0, 1.0
# Coupler point offset
PX, PY = 0.5, 0.3
# Target path
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
# Crank angles
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

BOUNDS = [(0.3, 2.5), (0.3, 2.5)]

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Analytical gradient (from Tutorial 13)

We redefine the analytical gradient inline since each notebook is self-contained.

In [ ]:
def coupler_point_and_grad(theta1, l2, l3):
    """Compute coupler point AND its gradient w.r.t. [l2, l3]."""
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ddx, ddy = ax - L0, ay
    d = np.sqrt(ddx**2 + ddy**2)
    
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None, None
    
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    sin_alpha = np.sin(alpha)
    if np.abs(sin_alpha) < 1e-15:
        return None, None
    
    beta = np.arctan2(ddy, ddx)
    theta3 = beta + alpha
    
    bx = L0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    rx, ry = bx - ax, by - ay
    r2 = rx**2 + ry**2
    
    theta2 = np.arctan2(ry, rx)
    c2, s2 = np.cos(theta2), np.sin(theta2)
    px = ax + PX * c2 - PY * s2
    py = ay + PX * s2 + PY * c2
    point = np.array([px, py])
    
    # Derivatives
    dca_dl2 = -l2 / (l3 * d)
    dca_dl3 = (l3**2 - d**2 + l2**2) / (2 * l3**2 * d)
    dalpha_dca = -1.0 / sin_alpha
    dt3_dl2 = dalpha_dca * dca_dl2
    dt3_dl3 = dalpha_dca * dca_dl3
    
    c3, s3 = np.cos(theta3), np.sin(theta3)
    dbx_dl2 = -l3 * s3 * dt3_dl2
    dby_dl2 =  l3 * c3 * dt3_dl2
    dbx_dl3 = c3 + (-l3 * s3) * dt3_dl3
    dby_dl3 = s3 + ( l3 * c3) * dt3_dl3
    
    dt2_dl2 = (-ry * dbx_dl2 + rx * dby_dl2) / r2
    dt2_dl3 = (-ry * dbx_dl3 + rx * dby_dl3) / r2
    
    dp_dt2 = np.array([-PX * s2 - PY * c2,
                        PX * c2 - PY * s2])
    grad = np.column_stack([dp_dt2 * dt2_dl2, dp_dt2 * dt2_dl3])
    return point, grad


def analytical_gradient(x):
    """Analytical gradient of the objective w.r.t. [l2, l3]."""
    l2, l3 = x
    grad = np.zeros(2)
    for i, theta1 in enumerate(THETAS):
        pt, dp = coupler_point_and_grad(theta1, l2, l3)
        if pt is None:
            return np.array([0.0, 0.0])
        residual = pt - TARGETS[i]
        grad += 2.0 * residual @ dp
    return grad

### The problem with pure approaches

Let's first demonstrate the weaknesses of using each method alone.

#### Pure BFGS from random starts: fast but inconsistent

BFGS converges quickly but is sensitive to the starting point. Different starts find different local minima.

In [ ]:
rng = np.random.default_rng(42)

n_starts = 20
bfgs_results = []
t0 = time.perf_counter()
for _ in range(n_starts):
    x0 = rng.uniform(0.3, 2.5, size=2)
    res = minimize(objective, x0, method='L-BFGS-B',
                   jac=analytical_gradient,
                   bounds=BOUNDS, options={'ftol': 1e-12})
    bfgs_results.append(res)
time_multistart = time.perf_counter() - t0

bfgs_fvals = [r.fun for r in bfgs_results]
best_bfgs = min(bfgs_results, key=lambda r: r.fun)

print(f'Multi-start BFGS ({n_starts} starts):')
print(f'  Best f:  {best_bfgs.fun:.6f}')
print(f'  Worst f: {max(bfgs_fvals):.6f}')
print(f'  Median f: {np.median(bfgs_fvals):.6f}')
print(f'  Time: {time_multistart*1000:.0f} ms')
print(f'  Total f evals: {sum(r.nfev for r in bfgs_results)}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(n_starts), bfgs_fvals, color='steelblue', edgecolor='k', lw=0.5)
ax.axhline(best_bfgs.fun, color='red', ls='--', lw=1, label='best')
ax.set_xlabel('Start index')
ax.set_ylabel(r'$f(\mathbf{x}^*)$')
ax.set_title('Pure BFGS from 20 random starts: inconsistent results')
ax.legend()
plt.tight_layout()

#### Pure DE: consistent but slow to converge precisely

In [ ]:
# Pure DE with 200 generations
de_nfev = [0]
def obj_de(x):
    de_nfev[0] += 1
    return objective(x)

t0 = time.perf_counter()
res_de_full = differential_evolution(
    obj_de, bounds=BOUNDS, seed=42,
    maxiter=200, popsize=15, tol=1e-15,
    mutation=(0.5, 1.0), recombination=0.7
)
time_de_full = time.perf_counter() - t0

print(f'Pure DE (200 generations):')
print(f'  f(x*) = {res_de_full.fun:.6f}')
print(f'  x*    = [{res_de_full.x[0]:.4f}, {res_de_full.x[1]:.4f}]')
print(f'  f evals: {de_nfev[0]}')
print(f'  Time: {time_de_full*1000:.0f} ms')

### The hybrid approach: DE then BFGS

Now the hybrid strategy:
1. Run DE for only ~30 generations (coarse global search)
2. Take the best result and polish it with BFGS + analytical gradient

In [ ]:
# Step 1: Coarse DE
hybrid_nfev = [0]
def obj_hybrid(x):
    hybrid_nfev[0] += 1
    return objective(x)

t0 = time.perf_counter()
res_de_coarse = differential_evolution(
    obj_hybrid, bounds=BOUNDS, seed=42,
    maxiter=30, popsize=15, tol=1e-15,
    mutation=(0.5, 1.0), recombination=0.7
)
de_feval_count = hybrid_nfev[0]

print(f'Step 1 — Coarse DE (30 gen):')
print(f'  f = {res_de_coarse.fun:.6f}')
print(f'  x = [{res_de_coarse.x[0]:.4f}, {res_de_coarse.x[1]:.4f}]')
print(f'  f evals: {de_feval_count}')

# Step 2: Polish with BFGS
hybrid_path = [res_de_coarse.x.copy()]
res_hybrid = minimize(
    obj_hybrid, res_de_coarse.x, method='L-BFGS-B',
    jac=analytical_gradient, bounds=BOUNDS,
    callback=lambda xk: hybrid_path.append(xk.copy()),
    options={'ftol': 1e-12}
)
hybrid_path = np.array(hybrid_path)
time_hybrid = time.perf_counter() - t0

print(f'\nStep 2 — BFGS polish:')
print(f'  f = {res_hybrid.fun:.6f}')
print(f'  x = [{res_hybrid.x[0]:.4f}, {res_hybrid.x[1]:.4f}]')
print(f'  BFGS iterations: {len(hybrid_path)-1}')
print(f'  Total f evals: {hybrid_nfev[0]}')
print(f'  Total time: {time_hybrid*1000:.0f} ms')

### Comparison table

Putting all methods side by side:

In [ ]:
print(f'{"Method":<28s} {"f(x*)":>10s} {"f evals":>10s} {"Time (ms)":>10s}')
print('-' * 62)
print(f'{"Pure BFGS (best of 20)":<28s} {best_bfgs.fun:10.6f} '
      f'{sum(r.nfev for r in bfgs_results):10d} '
      f'{time_multistart*1000:10.0f}')
print(f'{"Pure DE (200 gen)":<28s} {res_de_full.fun:10.6f} '
      f'{de_nfev[0]:10d} '
      f'{time_de_full*1000:10.0f}')
print(f'{"Hybrid DE(30) + BFGS":<28s} {res_hybrid.fun:10.6f} '
      f'{hybrid_nfev[0]:10d} '
      f'{time_hybrid*1000:10.0f}')

### Visualizing the hybrid strategy on the contour plot

Let's show the DE population converging, then the BFGS arrow polishing the result.

In [ ]:
# Build contour grid
l2v = np.linspace(0.3, 2.5, 40)
l3v = np.linspace(0.3, 2.5, 40)
L2g, L3g = np.meshgrid(l2v, l3v)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2g, L3g)

# Run DE again with callback to capture population snapshots
de_snapshots = []

# Use a wrapper that captures intermediate best via convergence list
de_best_history = []
de_convergence = []

def de_callback_fn(xk, convergence):
    de_best_history.append(xk.copy())
    de_convergence.append(objective(xk))

_ = differential_evolution(
    objective, bounds=BOUNDS, seed=42,
    maxiter=30, popsize=15, tol=1e-15,
    mutation=(0.5, 1.0), recombination=0.7,
    callback=de_callback_fn
)
de_best_history = np.array(de_best_history)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis')

# DE convergence path (best per generation)
if len(de_best_history) > 1:
    ax.plot(de_best_history[:, 0], de_best_history[:, 1],
            'w.-', ms=4, lw=1, alpha=0.7, label='DE best (per gen)')
    ax.plot(de_best_history[0, 0], de_best_history[0, 1],
            'wo', ms=8, zorder=5, label='DE start')

# BFGS polishing path
ax.plot(hybrid_path[:, 0], hybrid_path[:, 1],
        'r->', ms=8, lw=2.5, markevery=[-1], label='BFGS polish')

# Final optimum
ax.plot(res_hybrid.x[0], res_hybrid.x[1],
        'r*', ms=15, zorder=6, label=f'optimum (f={res_hybrid.fun:.4f})')

ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Hybrid strategy: DE explores, BFGS polishes')
ax.legend(fontsize=8)
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### Multi-start local as an alternative

Another common approach: run BFGS from many random starts and take the best. How does it compare to the hybrid?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: multi-start BFGS endpoints
ax = axes[0]
cf = ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis')
for r in bfgs_results:
    color = 'lime' if r.fun < best_bfgs.fun + 0.01 else 'white'
    ax.plot(r.x[0], r.x[1], 'o', color=color, ms=6,
            markeredgecolor='k', markeredgewidth=0.5)
ax.plot(best_bfgs.x[0], best_bfgs.x[1], 'r*', ms=15, zorder=6)
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title(f'Multi-start BFGS (20 starts)\nbest f = {best_bfgs.fun:.6f}')
plt.colorbar(cf, ax=ax, label=r'$\ln(1 + f)$')

# Right: hybrid
ax = axes[1]
cf = ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis')
if len(de_best_history) > 1:
    ax.plot(de_best_history[:, 0], de_best_history[:, 1],
            'w.-', ms=3, lw=1, alpha=0.7, label='DE path')
ax.plot(hybrid_path[:, 0], hybrid_path[:, 1],
        'r->', ms=8, lw=2.5, markevery=[-1], label='BFGS polish')
ax.plot(res_hybrid.x[0], res_hybrid.x[1], 'r*', ms=15, zorder=6)
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title(f'Hybrid DE(30) + BFGS\nf = {res_hybrid.fun:.6f}')
ax.legend(fontsize=8)
plt.colorbar(cf, ax=ax, label=r'$\ln(1 + f)$')

plt.tight_layout()

### Final visualization: the optimized mechanism

Let's see the best mechanism from the hybrid strategy in action.

In [ ]:
l2_opt, l3_opt = res_hybrid.x
mechanism = FourBar(L0, L1, l2_opt, l3_opt)
path_opt = np.array([coupler_point(t, l2_opt, l3_opt) for t in THETAS])

fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=12, label='targets')
ax.plot(path_opt[:, 0], path_opt[:, 1], 'b.-', lw=2, ms=8,
        label='optimized coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'Hybrid optimum: $\\ell_2$={l2_opt:.3f}, '
             f'$\\ell_3$={l3_opt:.3f}  |  f = {res_hybrid.fun:.4f}')
plt.tight_layout()

### Convergence of the hybrid approach over time

In [ ]:
# DE convergence (per generation)
hybrid_bfgs_costs = [objective(p) for p in hybrid_path]

# Combine DE + BFGS convergence into one plot
de_gens = list(range(len(de_convergence)))
bfgs_iters = list(range(len(de_convergence),
                        len(de_convergence) + len(hybrid_bfgs_costs)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(de_gens, de_convergence, 'g-o', ms=3, lw=2,
            label='Phase 1: DE (global)')
ax.semilogy(bfgs_iters, hybrid_bfgs_costs, 'r-s', ms=5, lw=2,
            label='Phase 2: BFGS (local)')
ax.axvline(len(de_convergence) - 0.5, color='k', ls=':', lw=1,
           label='switch point')
ax.set_xlabel('Step (DE generation / BFGS iteration)')
ax.set_ylabel(r'$f(\mathbf{x})$')
ax.set_title('Hybrid convergence: global exploration then local refinement')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

Notice the characteristic pattern: DE reduces the objective steadily but slowly, then BFGS takes over and drops it rapidly to the final precision. The switch point is where the hybrid pays off — DE has done its job of finding the right basin, and BFGS exploits the smooth local structure.

---

## Exercises

1. **Tuning the switch point.** Try running DE for 10, 20, 50, and 100 generations before switching to BFGS. Plot total function evaluations vs. final objective value. Is there a sweet spot?

2. **Alternative global methods.** Replace DE with `scipy.optimize.dual_annealing` in the hybrid pipeline. Compare total cost and final quality.

3. **Higher dimensions.** Add $\ell_1$, $P_x$, and $P_y$ as design variables (5 total). Does the hybrid advantage grow with dimensionality? You will need to extend the analytical gradient from Tutorial 13.

4. **Adaptive switching.** Instead of a fixed number of DE generations, implement a criterion that switches to BFGS when the DE population diversity drops below a threshold. Does this outperform the fixed-generation approach?